# Query Fabric Graph with GQL over REST

This developer-focused notebook uses the Contoso product review graph created by `infra/create-fabric-graph.py`.

Fabric exposes one stateless query API: send GQL in a JSON request to `executeQuery`. The same GQL can also be pasted into the Fabric Graph code editor for interactive development.

## 1. Install and import dependencies

Use the repository's `uv` environment as the notebook kernel. It already includes `azure-identity`, `httpx`, `python-dotenv`, and `ipykernel`; no inline installation or credential output is needed.

In [ ]:
import json
import os
from pathlib import Path

import httpx
from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv
from IPython.display import JSON, display

print("Dependencies imported")

Dependencies imported


## 2. Configure the endpoint and authentication

Load resource identifiers written to `.env` during provisioning. `AzureDeveloperCliCredential` obtains a delegated Fabric token without storing or displaying it in the notebook.

In [ ]:
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(repo_root / ".env", override=True)

fabric_tenant_id = os.environ["FABRIC_TENANT_ID"]
fabric_workspace_id = os.environ["FABRIC_WORKSPACE_ID"]
fabric_graph_id = os.environ["FABRIC_GRAPH_ID"]
fabric_scope = "https://api.fabric.microsoft.com/.default"
gql_endpoint = (
    "https://api.fabric.microsoft.com/v1/workspaces/"
    f"{fabric_workspace_id}/GraphModels/{fabric_graph_id}/executeQuery?preview=true"
)
credential = AzureDeveloperCliCredential(tenant_id=fabric_tenant_id)

print(f"Graph endpoint configured for workspace {fabric_workspace_id}")

Graph endpoint configured for workspace f200c1bc-e2ec-43b9-9308-be868f68753e


## 3. Define a reusable GQL query

This query mirrors the slide's graph pattern and keeps the node and edge variables visible. It finds negative feature mentions connected to product reviews.

In [ ]:
negative_mentions_query = """
MATCH (p:`Product`)-[:`HAS_REVIEW`]->(r:`Review`)-[m:`MENTIONS`]->(f:`Feature`)
FILTER m.sentiment = 'negative'
RETURN p.name AS productName,
       r.rating AS rating,
       f.featureName AS feature,
       m.sentiment AS sentiment,
       m.confidence AS confidence
ORDER BY confidence DESC
LIMIT 20
""".strip()

print(negative_mentions_query)

MATCH (p:`Product`)-[:`HAS_REVIEW`]->(r:`Review`)-[m:`MENTIONS`]->(f:`Feature`)

FILTER m.sentiment = 'negative'

RETURN p.name AS productName,

       r.rating AS rating,

       f.featureName AS feature,

       m.sentiment AS sentiment,

       m.confidence AS confidence

ORDER BY confidence DESC

LIMIT 20


## 4. Execute the query as GQL

For interactive development, paste `negative_mentions_query` into the Fabric Graph **Code editor** and select **Run query**. The code editor and API execute the same GQL against the same queryable graph; Fabric does not expose a second, non-HTTP developer endpoint.

## 5. Execute GQL through REST

The stateless API accepts `POST {"query": "..."}`. A successful HTTP response can still contain a GQL application error, so `execute_gql` checks both the HTTP status and the GQLSTATUS prefix.

In [ ]:
SUCCESS_STATUS_CLASSES = {"00", "01", "02", "03"}


class GQLQueryError(RuntimeError):
    """Raised when Fabric reports a GQL application-level failure."""


def execute_gql(
    query,
    *,
    http_client=None,
    access_token=None,
    timeout=httpx.Timeout(30.0, read=300.0),
):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a nonempty string")

    token = access_token or credential.get_token(fabric_scope).token
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    owns_client = http_client is None
    client = http_client or httpx.Client(timeout=timeout)
    try:
        response = client.post(
            gql_endpoint,
            headers=headers,
            json={"query": query},
        )
        response.raise_for_status()
        try:
            payload = response.json()
        except ValueError as exc:
            raise RuntimeError("Fabric returned a non-JSON response") from exc
    finally:
        if owns_client:
            client.close()

    status = payload.get("status")
    if not isinstance(status, dict) or not isinstance(status.get("code"), str):
        raise RuntimeError("Fabric response is missing a valid status object")
    if status["code"][:2] not in SUCCESS_STATUS_CLASSES:
        raise GQLQueryError(json.dumps(status, indent=2))

    return payload


negative_mentions_response = execute_gql(negative_mentions_query)
print(negative_mentions_response["status"]["description"])

note: successful completion


## 6. Inspect the response

Fabric returns an execution outcome containing `status` and, for tabular queries, a typed `result`. The rows in `result.data` are already compact JSON objects.

In [ ]:
def result_rows(payload):
    result = payload.get("result", {})
    if result.get("kind") == "NOTHING":
        return []
    if result.get("kind") != "TABLE" or not isinstance(result.get("data"), list):
        raise RuntimeError("Fabric response does not contain a tabular result")
    return result["data"]


print("Columns:")
display(JSON(negative_mentions_response["result"]["columns"]))
print("Rows:")
display(JSON(result_rows(negative_mentions_response)))

Columns:


<IPython.core.display.JSON object>

Rows:


<IPython.core.display.JSON object>

## 7. Handle errors and timeouts

`httpx` raises transport exceptions for authentication failures (`401`/`403`), rate limits (`429`), server failures, and timeouts. `execute_gql` separately raises `GQLQueryError` for application failures returned inside HTTP 200 and rejects malformed JSON or missing status objects.

Production clients can catch these categories independently:

```python
try:
    payload = execute_gql(query)
except httpx.TimeoutException:
    ...  # retry according to application policy
except httpx.HTTPStatusError as exc:
    ...  # inspect exc.response.status_code
except GQLQueryError:
    ...  # fix or report the GQL query
```

## 8. Run controlled query variants

The preview API currently accepts only a `query` field, not separately bound parameters. Never splice unrestricted user input into GQL. This example permits only known sentiment values and emits the corresponding literal from an allowlist.

In [ ]:
SENTIMENT_LITERALS = {
    "positive": "'positive'",
    "neutral": "'neutral'",
    "negative": "'negative'",
}


def feature_sentiment_query(sentiment):
    try:
        sentiment_literal = SENTIMENT_LITERALS[sentiment]
    except KeyError as exc:
        raise ValueError(f"Unsupported sentiment: {sentiment!r}") from exc

    return f"""
MATCH (r:`Review`)-[m:`MENTIONS`]->(f:`Feature`)
FILTER m.sentiment = {sentiment_literal}
LET featureName = f.featureName
RETURN featureName, count(m) AS mentions
GROUP BY featureName
ORDER BY mentions DESC
""".strip()


sentiment_results = {}
for sentiment in SENTIMENT_LITERALS:
    response = execute_gql(feature_sentiment_query(sentiment))
    sentiment_results[sentiment] = result_rows(response)

display(JSON(sentiment_results))

<IPython.core.display.JSON object>

## 9. Paginate query results

Fabric GQL supports ordered `OFFSET`/`LIMIT` pagination rather than a server cursor in the current API. Always use a stable `ORDER BY`, validate numeric controls locally, and enforce a maximum page count.

In [ ]:
def product_page_query(offset, page_size):
    if type(offset) is not int or offset < 0:
        raise ValueError("offset must be a nonnegative integer")
    if type(page_size) is not int or not 1 <= page_size <= 100:
        raise ValueError("page_size must be an integer from 1 through 100")

    return f"""
MATCH (p:`Product`)
RETURN p.sku AS sku, p.name AS productName
ORDER BY sku
OFFSET {offset}
LIMIT {page_size}
""".strip()


def paginate_products(*, page_size=10, max_pages=5, executor=execute_gql):
    if type(max_pages) is not int or max_pages < 1:
        raise ValueError("max_pages must be a positive integer")

    combined_rows = []
    for page_number in range(max_pages):
        query = product_page_query(page_number * page_size, page_size)
        page_rows = result_rows(executor(query))
        combined_rows.extend(page_rows)
        if len(page_rows) < page_size:
            break

    return combined_rows


products = paginate_products(page_size=10, max_pages=3)
print(f"Retrieved {len(products)} products")
display(JSON(products))

Retrieved 30 products


<IPython.core.display.JSON object>

## 10. Validate the query client

`httpx.MockTransport` exercises request construction and failure handling without calling Fabric. These lightweight assertions verify the JSON payload, bearer header, successful parsing, GQL application errors, HTTP errors, malformed responses, timeouts, and pagination limits.

In [ ]:
def response_payload(rows=None, code="00000"):
    return {
        "status": {"code": code, "description": "test status", "diagnostics": {}},
        "result": {
            "kind": "TABLE",
            "columns": [],
            "data": rows or [],
        },
    }


captured_request = {}


def success_handler(request):
    captured_request["authorization"] = request.headers["Authorization"]
    captured_request["payload"] = json.loads(request.content)
    return httpx.Response(200, json=response_payload([{"value": 1}]))


with httpx.Client(transport=httpx.MockTransport(success_handler)) as mock_client:
    payload = execute_gql(
        "MATCH (n) RETURN n LIMIT 1",
        http_client=mock_client,
        access_token="test-token",
    )

assert captured_request == {
    "authorization": "Bearer test-token",
    "payload": {"query": "MATCH (n) RETURN n LIMIT 1"},
}
assert result_rows(payload) == [{"value": 1}]


def assert_raises(expected_exception, handler):
    with httpx.Client(transport=httpx.MockTransport(handler)) as mock_client:
        try:
            execute_gql("MATCH (n) RETURN n", http_client=mock_client, access_token="test-token")
        except expected_exception:
            return
    raise AssertionError(f"Expected {expected_exception.__name__}")


assert_raises(
    GQLQueryError,
    lambda _request: httpx.Response(200, json=response_payload(code="42000")),
)
assert_raises(
    httpx.HTTPStatusError,
    lambda _request: httpx.Response(429, json={"error": "rate limited"}),
)
assert_raises(
    RuntimeError,
    lambda _request: httpx.Response(200, text="not json"),
)


def timeout_handler(_request):
    raise httpx.ReadTimeout("test timeout", request=_request)


assert_raises(httpx.ReadTimeout, timeout_handler)

page_calls = []


def fake_executor(query):
    page_calls.append(query)
    offset = len(page_calls) - 1
    return response_payload([{"sku": f"SKU-{offset}"}])


assert paginate_products(page_size=1, max_pages=2, executor=fake_executor) == [
    {"sku": "SKU-0"},
    {"sku": "SKU-1"},
]
assert len(page_calls) == 2

print("All query client checks passed")

All query client checks passed


## Recap

- `MATCH` describes the node-edge pattern to traverse.
- `FILTER` keeps only rows matching a predicate.
- `LET` names a property or computed value for later statements.
- `RETURN` projects properties and aggregates into a result table.
- `GROUP BY` summarizes matched rows.
- `ORDER BY`, `OFFSET`, and `LIMIT` provide deterministic ranking and pagination.
- The REST API sends the GQL as `{"query": "..."}` and requires checks at both the HTTP and GQLSTATUS layers.